In [36]:
import numpy as np
import pandas as pd
import random

In [37]:
pd.set_option('display.max_columns', None)

In [38]:
dataset_workout = pd.read_csv('../../data/dataset_workout.csv')

In [39]:
df = pd.read_csv('../../data/dataset_users.csv')
df_weekly_progress = pd.read_csv('../../data/dataset_userprogress.csv')

df_merged = pd.merge(
    df,                       # Tabel kiri (Data Profil)
    df_weekly_progress,       # Tabel kanan (Data Mingguan)
    on='User_ID',             # Kunci Penghubung
    how='left'                # Jenis Join
)

In [40]:
SPORTS_COLUMNS = [
    'Badminton', 'Football', 'Basketball', 
    'Volleyball', 'Swim'
]

In [41]:
MUSCLE_GROUPS = {
    'Chest': ['pectorals', 'chest', 'serratus', 'push up', 'press'],
    'Shoulders': ['delts', 'shoulders', 'rotator', 'press', 'raise'],
    'Triceps': ['triceps', 'extension', 'dip', 'pushdown', 'skullcrusher'],
    'Back': ['lats', 'latissimus', 'trapezius', 'row', 'pull', 'chin', 'superman'],
    'Biceps': ['biceps', 'curl', 'hammer'],
    'Quads': ['quads', 'quadriceps', 'squat', 'lunge', 'step up', 'leg press', 'goblet'], 
    'Hamstrings': ['hamstrings', 'curl', 'glute ham', 'deadlift', 'hinge', 'good morning'], 
    'Glutes': ['glutes', 'hip', 'butt', 'bridge', 'kickback', 'thrust'], 
    'Calves': ['calves', 'raise', 'jump'],
    'Abs': ['abs', 'abdominal', 'core', 'crunch', 'plank', 'sit up', 'leg raise', 'twist'],
    'Cardio': ['cardio', 'run', 'jump', 'burpee', 'climber', 'jack', 'sprint'],
    
    # Fallback Categories
    'Push_General': ['chest', 'shoulders', 'triceps', 'push'],
    'Pull_General': ['back', 'biceps', 'pull'],
    'Legs_General': ['legs', 'lower body', 'squat', 'lunge'],
}

In [42]:
def clean_string_data(text):
    """
    Fungsi untuk mengubah "['cable']" menjadi "cable"
    Sangat berguna karena CSV membaca list sebagai teks biasa.
    """
    if pd.isna(text): 
        return "None"
    # Menghapus kurung siku dan tanda kutip satu per satu
    cleaned = str(text).replace("[", "").replace("]", "").replace("'", "")
    return cleaned.strip()

In [43]:
# --- HELPER 1: BERSIHKAN STRING ---
def clean_string_data(text):
    if pd.isna(text) or str(text).lower() == 'nan': return "None"
    return str(text).replace("[", "").replace("]", "").replace("'", "").strip()

# --- HELPER 2: TENTUKAN KATEGORI LATIHAN (TASK 2) ---
def get_exercise_category(ex_name, muscle_group, env):
    """
    Menentukan apakah ini Weightlifting (Strength), Cardio, atau Sport.
    """
    ex_name = str(ex_name).lower()
    muscle_group = str(muscle_group).lower()
    env = str(env)
    
    # 1. Sport
    if env == 'Other' or 'sport' in env.lower():
        return 'Sport'
    
    # 2. Cardio
    if 'cardio' in muscle_group or 'cardio' in ex_name:
        return 'Cardio'
    
    # 3. Strength / Weightlifting
    # Default untuk latihan beban atau bodyweight exercises
    return 'Strength'

# --- HELPER 3: SMART SEARCH DENGAN EXCLUSION ---
def get_exercises_smart(df_exercises, target_group, target_env, exclude_names=[], limit=1):
    # Filter Environment
    df_env = df_exercises[df_exercises['Environment'] == target_env]
    if df_env.empty: return pd.DataFrame()

    keywords = MUSCLE_GROUPS.get(target_group, [])
    
    # Filter Keyword
    mask_specific = df_env['targetMuscles'].astype(str).apply(
        lambda x: any(k.lower() in x.lower() for k in keywords)
    )
    
    # Exclude yang sudah dipakai
    mask_unused = ~df_env['name'].isin(exclude_names)
    
    # Coba cari yang spesifik & belum dipakai
    result = df_env[mask_specific & mask_unused]
    
    # FALLBACK LOGIC
    if result.empty:
        # Mapping Fallback
        fallback_map = {
            'Quads': 'Legs_General', 'Hamstrings': 'Legs_General', 'Glutes': 'Legs_General', 'Calves': 'Legs_General',
            'Chest': 'Push_General', 'Shoulders': 'Push_General', 'Triceps': 'Push_General',
            'Back': 'Pull_General', 'Biceps': 'Pull_General'
        }
        if target_group in fallback_map:
            gen_group = fallback_map[target_group]
            gen_keywords = MUSCLE_GROUPS.get(gen_group, [])
            mask_general = df_env['targetMuscles'].astype(str).apply(
                lambda x: any(k.lower() in x.lower() for k in gen_keywords)
            )
            result = df_env[mask_general & mask_unused]
    
    # PANIC MODE (Ambil Duplikat jika terpaksa)
    if result.empty:
        result = df_env[mask_specific]
        
    if len(result) > 0:
        return result.sample(n=min(limit, len(result)), replace=False)
    
    return pd.DataFrame()

In [44]:
import random
import pandas as pd
import numpy as np

# Pastikan SPORTS_COLUMNS sudah didefinisikan sebelumnya
# SPORTS_COLUMNS = [...] 

def generate_weekly_plan(user_row, df_exercises):
    user_id = user_row['User_ID']
    freq = user_row['Workout_Frequency_x'] 
    duration = user_row['Average_Duration_Minutes_x']
    goal = user_row['Goal_x']
    
    # --- A. VARIATION: RANDOM ENVIRONMENT PER USER ---
    main_env = random.choice(['Gym', 'Home'])
    
    # --- B. HOBI (SPORT) CHECK ---
    user_sports = []
    for col in SPORTS_COLUMNS:
        if user_row.get(col, 0) == 1:
            sport_name = col.replace('_x', '').replace('_', ' ')
            user_sports.append(sport_name)
            
    # --- C. MINIMAL 5 GERAKAN ---
    target_exercises = max(5, int(duration / 7))
    target_exercises = min(target_exercises, 12)
    
    weekly_schedule = []

    # ============================================================
    # [BARU] LOGIC VARIATION STYLE (Lebih Kompleks & Variatif)
    # ============================================================
    # Kita tentukan "Kepribadian Latihan" user ini secara acak berdasarkan Goal
    workout_style = 'Balanced' # Default (Campur)

    if goal == 'Weight Loss':
        rand = random.random()
        if rand < 0.4:
            workout_style = 'Cardio Focused'    # 40% Fokus Bakar Lemak (Dominan Cardio)
        elif rand < 0.7:
            workout_style = 'Strength Focused'  # 30% Fokus Angkat Beban (Fat loss via muscle)
        else:
            workout_style = 'Balanced'          # 30% Hybrid
            
    elif goal == 'Muscle Gain':
        if random.random() < 0.6:
            workout_style = 'Pure Strength'     # 60% Fokus Hipertrofi (Minim Cardio)
        else:
            workout_style = 'Hybrid Athlete'    # 40% Angkat Beban + Hobi Sport
            
    elif goal == 'Maintain':
        workout_style = 'Balanced'              # Maintain biasanya seimbang
    # ============================================================
    
    # --- D. SPLIT MAP (TETAP SAMA) ---
    schedule_map = {}
    
    if freq <= 2:
        schedule_map = {
            1: {'Theme': 'Full Body Push & Cardio', 'Focus': ['Quads', 'Chest', 'Shoulders', 'Triceps', 'Cardio']}, 
            2: {'Theme': 'Full Body Pull & Core', 'Focus': ['Hamstrings', 'Back', 'Biceps', 'Glutes', 'Abs']}}
    elif freq == 3:
        schedule_map = {
            1: {'Theme': 'Push & Burn', 'Focus': ['Chest', 'Shoulders', 'Triceps', 'Cardio']}, 
            2: {'Theme': 'Pull & Core', 'Focus': ['Back', 'Biceps', 'Traps', 'Abs']}, 
            3: {'Theme': 'Leg Power', 'Focus': ['Quads', 'Hamstrings', 'Glutes', 'Calves']}}
    elif freq == 4:
        schedule_map = {
            1: {'Theme': 'Upper Strength', 'Focus': ['Chest', 'Back', 'Shoulders', 'Abs']}, 
            2: {'Theme': 'Lower Quads & Cardio', 'Focus': ['Quads', 'Calves', 'Cardio', 'Abs']}, 
            3: {'Theme': 'Upper Pump', 'Focus': ['Biceps', 'Triceps', 'Chest', 'Back']}, 
            4: {'Theme': 'Lower Hams & Glutes', 'Focus': ['Hamstrings', 'Glutes', 'Calves']}}
    else: 
        schedule_map = {
            1: {'Theme': 'Chest & Back', 'Focus': ['Chest', 'Back', 'Abs']}, 
            2: {'Theme': 'Legs & Cardio', 'Focus': ['Quads', 'Hamstrings', 'Cardio']}, 
            3: {'Theme': 'Shoulders & Arms', 'Focus': ['Shoulders', 'Biceps', 'Triceps']}, 
            4: {'Theme': 'Lower Glute Focus', 'Focus': ['Glutes', 'Calves', 'Abs']}, 
            5: {'Theme': 'Upper Body Mix', 'Focus': ['Chest', 'Shoulders', 'Back']}, 
            6: {'Theme': 'Cardio & Core', 'Focus': ['Abs', 'Cardio', 'Obliques']}}

    # --- E. GENERATE DAYS ---
    for day_num, config in schedule_map.items():
        if day_num > freq: break
            
        theme = config['Theme']
        day_muscles = list(config['Focus'])
        used_exercises_today = [] 
        
        # ============================================================
        # [MODIFIKASI] LOGIC OVERRIDE & INJEKSI BERDASARKAN STYLE
        # ============================================================
        
        # 1. KASUS KHUSUS: CARDIO FOCUSED (Weight Loss Extreme)
        # Override jadwal jadi full cardio/interval
        if workout_style == 'Cardio Focused':
            theme = f"Cardio Burn Day {day_num}"
            day_muscles = ['Cardio', 'Cardio', 'Abs', 'Cardio', 'Cardio', 'Abs']
        
        # 2. KASUS LAINNYA (Strength, Hybrid, Balanced)
        # Disini kita atur peluang munculnya "Bonus Cardio/Sport" di akhir sesi
        else:
            add_bonus_cardio = False
            
            # Cek probabilitas berdasarkan Style User
            if workout_style == 'Pure Strength':
                # Muscle Gain yg fokus otot, jarang cardio tpi BISA (misal 20% chance)
                # biar ttp ada variasi, ga kaku bgt
                if random.random() < 0.2: add_bonus_cardio = True
                
            elif workout_style == 'Strength Focused':
                # Weight Loss yg fokus beban, cardio ada tpi secukupnya (50%)
                if random.random() < 0.5: add_bonus_cardio = True
                
            elif workout_style == 'Hybrid Athlete':
                # Muscle Gain yg suka olahraga (Basket dll) -> Sering (80%)
                if len(user_sports) > 0 and random.random() < 0.8: add_bonus_cardio = True
                elif random.random() < 0.4: add_bonus_cardio = True
                
            elif workout_style == 'Balanced':
                # Maintain / Umum -> Cukup sering (60%)
                if random.random() < 0.6: add_bonus_cardio = True
            
            # Eksekusi penambahan
            if add_bonus_cardio and 'Cardio' not in day_muscles:
                day_muscles.append('Cardio')

        # --- DISTRIBUSI SLOT ---
        num_groups = len(day_muscles)
        base_slot = target_exercises // num_groups
        remainder = target_exercises % num_groups
        
        for i, muscle in enumerate(day_muscles):
            my_limit = base_slot
            if i < remainder: my_limit += 1
            my_limit = max(1, my_limit)
            
            # === HANDLING CARDIO / SPORT ===
            if muscle == 'Cardio':
                # Tentukan apakah pakai SPORT EXTERNAL (Basket/Renang) atau CARDIO GYM?
                use_sport = False
                
                if len(user_sports) > 0:
                    # Logic Pemilihan Sport berdasarkan Style
                    if workout_style == 'Cardio Focused':
                         # 50% Sport, 50% Lari/HIIT (Variasi Weight Loss)
                         if random.random() < 0.5: use_sport = True
                         
                    elif workout_style == 'Pure Strength':
                        # Jarang main sport, tpi kalau pas dapet slot cardio,
                        # prioritasin sport (krn lbh fun drpd lari di treadmill)
                        use_sport = True 
                        
                    elif workout_style == 'Hybrid Athlete' or workout_style == 'Balanced':
                        # User tipe ini SANGAT suka sport
                        if random.random() < 0.8: use_sport = True
                        
                    elif workout_style == 'Strength Focused':
                        # Weight loss angkat beban, sport oke buat bakar kalori
                        if random.random() < 0.6: use_sport = True
                
                if use_sport:
                    sport = random.choice(user_sports)
                    category = 'Sport' 
                    weekly_schedule.append({
                        'User_ID': user_id, 'Day': f"Day {day_num} - {theme}",
                        'Muscle Group': 'Cardio', 
                        'Exercise Name': sport,
                        'Equipment': "['Sport Equipment']", 'SecondaryMuscles': "['None']",
                        'Sets': 1, 'Reps': f"{random.randint(30, 60)} Mins",
                        'Instructions': f"Play {sport} for active recovery/endurance.", 
                        'Environment': 'Other', # Sesuai request
                        'Workout_Type': category 
                    })
                else:
                    # Cardio DB (Sesuai Env User)
                    exercises = get_exercises_smart(df_exercises, 'Cardio', main_env, used_exercises_today, limit=1)
                    for _, ex in exercises.iterrows():
                        category = get_exercise_category(ex['name'], 'cardio', ex['Environment'])
                        weekly_schedule.append({
                            'User_ID': user_id, 'Day': f"Day {day_num} - {theme}",
                            'Muscle Group': 'Cardio', 
                            'Exercise Name': ex['name'],
                            'Equipment': clean_string_data(ex['equipments']),
                            'SecondaryMuscles': clean_string_data(ex.get('secondaryMuscles', "['None']")),
                            'Sets': 1, 'Reps': f"{random.randint(15, 30)} Mins",
                            'Instructions': clean_string_data(ex['instructions']), 
                            'Environment': ex['Environment'],
                            'Workout_Type': category
                        })
                        used_exercises_today.append(ex['name'])

            # === HANDLING STRENGTH ===
            else:
                exercises = get_exercises_smart(df_exercises, muscle, main_env, used_exercises_today, limit=my_limit)
                
                for _, ex in exercises.iterrows():
                    # Variasi Reps berdasarkan Style juga
                    if goal == 'Muscle Gain': 
                        sets, reps = 3, "8-12"
                    elif goal == 'Weight Loss': 
                        if workout_style == 'Strength Focused':
                            sets, reps = 3, "8-10" # Fokus beban berat dikit buat maintain otot
                        else:
                            sets, reps = 4, "12-15" # High rep burn
                    else: 
                        sets, reps = 3, "10-12"
                    
                    category = get_exercise_category(ex['name'], muscle, ex['Environment'])
                    
                    weekly_schedule.append({
                        'User_ID': user_id, 'Day': f"Day {day_num} - {theme}",
                        'Muscle Group': muscle, 
                        'Exercise Name': ex['name'],
                        'Equipment': clean_string_data(ex['equipments']),
                        'SecondaryMuscles': clean_string_data(ex.get('secondaryMuscles', "['None']")),
                        'Sets': sets, 'Reps': reps,
                        'Instructions': clean_string_data(ex['instructions']), 
                        'Environment': ex['Environment'],
                        'Workout_Type': category 
                    })
                    used_exercises_today.append(ex['name'])

    return pd.DataFrame(weekly_schedule)

In [45]:
# --- EXECUTE ---
all_user_plans = []
unique_users_df = df_merged.drop_duplicates(subset=['User_ID'])

print("Sedang membuat jadwal variatif (Min 5 Gerakan, Random Env)...")

for index, user_row in unique_users_df.iterrows():
    # Panggil fungsi
    my_plan = generate_weekly_plan(user_row, dataset_workout) # Pastikan dataset_workout yang sudah ditambah 25 data
    
    if not my_plan.empty:
        all_user_plans.append(my_plan)

if len(all_user_plans) > 0:
    df_final = pd.concat(all_user_plans, ignore_index=True)
    print(f"Sukses! Total baris jadwal: {len(df_final)}")
    
    # Preview untuk cek kolom baru dan variasi
    cols = ['User_ID', 'Day', 'Exercise Name', 'Environment', 'Workout_Type']
    print("\nPreview Data:")
    display(df_final[cols].head(15))
else:
    print("Warning: Tidak ada jadwal.")

Sedang membuat jadwal variatif (Min 5 Gerakan, Random Env)...
Sukses! Total baris jadwal: 13601

Preview Data:


,User_ID,Day,Exercise Name,Environment,Workout_Type
0,1,Day 1 - Push & Burn,smith incline bench press,Gym,Strength
1,1,Day 1 - Push & Burn,barbell incline bench press,Gym,Strength
2,1,Day 1 - Push & Burn,dumbbell one arm upright row,Gym,Strength
3,1,Day 1 - Push & Burn,face pull,Gym,Strength
4,1,Day 1 - Push & Burn,tricep dip on chair,Gym,Strength
5,1,Day 1 - Push & Burn,Assault Bike,Gym,Cardio
6,1,Day 2 - Pull & Core,lat pullover,Gym,Strength
7,1,Day 2 - Pull & Core,lever front pulldown,Gym,Strength
8,1,Day 2 - Pull & Core,cable seated curl,Gym,Strength
9,1,Day 2 - Pull & Core,dumbbell one arm hammer preacher curl,Gym,Strength


In [46]:
df_final_workout = pd.concat(all_user_plans, ignore_index=True)
df_final = pd.merge(
    df_merged,      # Data Mingguan (Week 0-12)
    df_final_workout,   # Data Latihan (Day 1-X)
    on='User_ID',   # Disambung pake User ID
    how='left'      # Left Join
)

print(f"Sukses! df_final berhasil dibuat dengan ukuran: {df_final.shape}")

Sukses! df_final berhasil dibuat dengan ukuran: (176813, 60)


In [47]:
# Daftar kolom yang mau dibersihkan (Olahraga & Profil)
cols_to_clean = [
    'Badminton', 'Football', 'Basketball', 'Volleyball', 'Swim',
]

for col in cols_to_clean:
    col_x = f"{col}_x"
    col_y = f"{col}_y"
    
    # Cek apakah kedua versi (_x dan _y) ada di dataframe
    if col_x in df_final.columns and col_y in df_final.columns:
        # Kita pakai yang _x sebagai data utama, lalu rename jadi nama asli
        df_final.rename(columns={col_x: col}, inplace=True)
        # Hapus yang _y karena duplikat
        df_final.drop(columns=[col_y], inplace=True)
        
    # Jaga-jaga kalau cuma ada _x (misal sisa merge sebelumnya)
    elif col_x in df_final.columns:
        df_final.rename(columns={col_x: col}, inplace=True)

In [48]:
# --- SIMPAN DATA FINAL KE CSV ---

output_file = '../../data/dataset_final.csv'

# Langsung simpan

df_final.to_csv(output_file, index=False)

print(f"Data berhasil disimpan ke: {output_file}")
print("Total baris:", len(df_final))

Data berhasil disimpan ke: ../../data/dataset_final.csv
Total baris: 176813
